# Pawnee National Grassland Land Swap
- **Objective:** For this notebook, the observations of 5 grasses, including blue grama(Bouteloua gracilis), buffalograss(Bouteloua dactyloides), sideoats grama(Bouteloua curtipendula), western wheatgrass(Pascopyrum smithii), and needle-and-thread (Hesperostipa comata), are pulled from [GBIF](https://www.gbif.org/) for each parcel within the Pawnee National Grassland boundary. A "ecological value" for these species will be generated for each parcel, and in the `parcel_matrix` notebook these ecological values will be appended to identify best land swaps.
- **Author:** Kayleigh Ward
- **Author:**
- **Code review and/or edits:** Kayleigh Ward
- **Date:** April 9, 2026
- **Last date of revision:** April 9, 2026

---
### 🛠️ Prerequisites & Setup
* **Libraries:** ex: `pandas`, `scikit-learn`, `matplotlib`
* **Data Sources:** ex: `/data/raw/dataset.csv`
* **Related Notebooks:** Must run the `boundaries` notebook prior to running this notebook and pull stored values.

### 🏗️ Methodology
1.  **Data Loading:** Login to GBIF and then download one GBIF species file, extract it, and append it to the GBIF grasses DataFrame. 
2.  **Cleaning:** Clip the GBIF grasses observations to the Pawnee National Grassland boundary
3.  **Visualization:** Plot the species distribution for both each grass to confirm that the clip was successful.

### ⚡ Troubleshooting/Notes
* Example Note: If data updates, run `01_refresh.ipynb` first.

# Libraries

In [ ]:
### import libraries
import os
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

### data handling 
import pandas as pd
import geopandas as gpd

### web requests / data download
import requests

### geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

### GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species
from getpass import getpass

# Primary Directory

In [ ]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

# Secondary Directories from 01_boundaries notebook

In [ ]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Western Pawnee
boundary_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')

### MASTER
master_bound_west = os.path.join(boundary_dir_west, 'master_boundary')
master_bound_west_path = os.path.join(master_bound_west, "pawnee_master_west.shp")
master_bound_west_gdf = gpd.read_file(master_bound_west_path)

### FEDERAL
federal_bound_west = os.path.join(boundary_dir_west, 'federal_boundary')
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)

### STATE
state_bound_west = os.path.join(boundary_dir_west, 'state_boundary')
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')
state_bound_west_gdf = gpd.read_file(state_bound_west_path)

### PARCEL
parcel_bound_west = os.path.join(boundary_dir_west, 'parcel_boundary')
parcel_bound_west_path = os.path.join(parcel_bound_west, 'pawnee_parcel_west.shp')
parcel_bound_west_gdf = gpd.read_file(parcel_bound_west_path)

### output directory for clipped GBIF points
gbif_clipped_dir = os.path.join(data_dir, "gbif_clipped")
os.makedirs(gbif_clipped_dir, exist_ok=True)

# GBIF Login for data download

In [ ]:
### reset credentials
reset_credentials = False

### make dictionary for GBIF username and pass
credentials = dict(
    GBIF_USER=(input, 'GBIF username:'),
    GBIF_PWD=(getpass, 'GBIF password'),
    GBIF_EMAIL=(input, 'GBIF email'),
)

### loop through credentials and enter them
for env_variable, (prompt_func, prompt_text) in credentials.items():

    if reset_credentials and (env_variable in os.environ):
        os.environ.pop(env_variable)

    if not env_variable in os.environ:
        os.environ[env_variable] = prompt_func(prompt_text)

## Download GBIF data for Pawnee National Grassland grasses

1. Create a helper function to pull data from GBIF
2. Define the species to download
3. Download data for the following grass species:
- blue grama(Bouteloua gracilis)
- buffalograss(Bouteloua dactyloides)
- sideoats grama(Bouteloua curtipendula)
- western wheatgrass(Pascopyrum smithii)
- needle-and-thread (Hesperostipa comata)

In [ ]:
### function to download GBIF data for five grasses
def download_gbif_grass_species(species_name, folder_name, data_dir):
    """
    Download one GBIF species file, extract it, and read it into a DataFrame.
    Returns:
        df, csv_path, species_key
    """
    ### search species in GBIF backbone
    species_info = species.name_backbone(name=species_name)
    species_key = species_info["usageKey"]
    print(f"{species_name}: {species_key}")

    ### set species directory
    gbif_grasses_dir = os.path.join(data_dir, folder_name)
    os.makedirs(gbif_grasses_dir, exist_ok=True)

    ### look for an extracted occurrence csv first
    existing_csvs = glob(os.path.join(gbif_grasses_dir, "*.csv"))
    if existing_csvs:
        gbif_grasses_path = existing_csvs[0]
        gbif_grasses_df = pd.read_csv(gbif_grasses_path, sep="\t", low_memory=False)
        return gbif_grasses_df, gbif_grasses_path, species_key

    ### submit query
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ])
    download_key = gbif_query[0]

    ### wait for the download to build
    status = occ.download_meta(download_key)["status"]
    while status != "SUCCEEDED":
        if status in {"CANCELLED", "KILLED", "FAILED"}:
            raise RuntimeError(f"GBIF download failed for {species_name}: {status}")
        time.sleep(5)
        status = occ.download_meta(download_key)["status"]

    ### download zipped Darwin Core archive
    occ.download_get(download_key, path=gbif_grasses_dir)

    ### find and unzip the download
    zip_matches = glob(os.path.join(gbif_grasses_dir, "*.zip"))
    if not zip_matches:
        raise FileNotFoundError(f"No GBIF zip download found for {species_name}")

    zip_path = zip_matches[0]
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(gbif_grasses_dir)

    ### locate occurrence table
    csv_candidates = glob(os.path.join(gbif_grasses_dir, "*.csv")) + glob(os.path.join(gbif_grasses_dir, "*.txt"))
    if not csv_candidates:
        raise FileNotFoundError(f"No extracted occurrence table found for {species_name}")

    gbif_grasses_path = csv_candidates[0]
    gbif_grasses_df = pd.read_csv(gbif_grasses_path, sep="\t", low_memory=False)

    return gbif_grasses_df, gbif_grasses_path, species_key


In [ ]:
### define the species to download
species_to_download = [
    {
        "species_name": "Bouteloua gracilis",
        "folder_name": "gbif_blue_grama"
    },
    {
        "species_name": "Bouteloua dactyloides",
        "folder_name": "gbif_buffalograss"
    }
    {
        "species_name": "Bouteloua curtipendula",
        "folder_name": "gbif_sideoats_grama"
    }
       {
        "species_name": "Pascopyrum smithii",
        "folder_name": "gbif_western_wheatgrass"
    }
       {
        "species_name": "Hesperostipa comata",
        "folder_name": "gbif_needle_and_thread"
    }
]

### create an empty dictionary
gbif_grasses_data = {}

### loop through the download_gbif_species download
for item in species_to_download:
    gbif_grasses_df, gbif_grasses_path, species_key = download_gbif_grass_species(
        species_name=item["species_name"],
        folder_name=item["folder_name"],
        data_dir=data_dir
    )

    gbif_grasses_data[item["species_name"]] = {
        "df": gbif_grasses_df,
        "path": gbif_grasses_path,
        "species_key": species_key
    }

    print(f"\nLoaded {item['species_name']}")
    print(f"Path: {gbif_grasses_path}")
    print(gbif_grasses_df.head())

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1631097267.py, line 7)

# Save the dataframe for each grass species
- gbif_blue_grama_df
- gbif_buffalograss_df
- gbif_sideoats_grama_df
- gbif_western_wheatgrass_df
- gbif_needle_and_thread_df

In [ ]:
#save the five grass species as their own dataframes

gbif_blue_grama_df = gbif_grasses_data["Bouteloua gracilis"]["df"]

gbif_buffalograss_df = gbif_grasses_data["Bouteloua dactyloides"]["df"]

gbif_sideoats_grama_df = gbif_grasses_data["Bouteloua curtipendula"]["df"]

gbif_western_wheatgrass_df = gbif_grasses_data["Pascopyrum smithii"]["df"]

gbif_needle_and_thread_df = gbif_grasses_data["Hesperostipa comata"]["df"]

# Convert to geodataframes for mapping

In [ ]:
# Make these spatial data frames (geodataframes)
for species_name, species_data in gbif_grasses_data.items():
    gbif_grasses_df = species_data["df"].dropna(
        subset=["decimalLongitude", "decimalLatitude"]
    ).copy()

    gbif_grasses_gdf = gpd.GeoDataFrame(
        gbif_grasses_df,
        geometry=gpd.points_from_xy(
            gbif_grasses_df["decimalLongitude"],
            gbif_grasses_df["decimalLatitude"]
        ),
        crs="EPSG:4326"
    )

    gbif_grasses_data[species_name]["gdf"] = gbif_grasses_gdf

    print(f"\nCreated GeoDataFrame for {species_name}")
    print(gbif_grasses_gdf.head())

# Save each grass species geodataframe

In [ ]:
### save the grasses geodataframes

gbif_blue_grama_gdf = gbif_grasses_df["Bouteloua gracilis"]["gdf"]

gbif_buffalograss_gdf = gbif_grasses_df["Bouteloua dactyloides"]["gdf"]

gbif_sideoats_grama_gdf = gbif_grasses_df["Bouteloua curtipendula"]["gdf"]

gbif_western_wheatgrass_gdf = gbif_grasses_df["Pascopyrum smithii"]["gdf"]

gbif_needle_and_thread_gdf = gbif_grasses_df["Hesperostipa comata"]["gdf"]

# Composite them into one geodataframe

In [ ]:
# combine into one gdf
gbif_grasses_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            gbif_blue_grama_gdf,
            gbif_buffalograss_gdf,
            gbif_sideoats_grama_gdf,
            gbif_western_wheatgrass_gdf,
            gbif_needle_and_thread_gdf
        ],
        ignore_index=True
    ),
    geometry="geometry",
    crs="EPSG:4326"
)

# Plot the composite geodataframe

In [ ]:
# make a preliminary plot of the data
gbif_grasses_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Five grass species occurrences (GBIF)",
    frame_width=700,
    size=35
)

In [ ]:
# make a preliminary plot of the data with the Western Pawnee boundary
grass_points = gbif_grasses_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Five grass species occurrences (GBIF) at Western Pawnee",
    frame_width=700,
    size=35
)

grass_boundary = master_bound_west_gdf.to_crs("EPSG:4326").hvplot(
    geo=True,
    color=None,
    line_color="black",
    line_width=2
)

grass_points * grass_boundary

In [ ]:
# clip in a shared CRS
master_bound_west_wgs84 = master_bound_west_gdf.to_crs("EPSG:4326")
gbif_grasses_clipped = gpd.clip(gbif_grasses_gdf, master_bound_west_wgs84)

print(f"Original records: {len(gbif_grasses_gdf)}")
print(f"Clipped records: {len(gbif_grasses_clipped)}")
gbif_grasses_clipped.head()

In [ ]:
# save the clipped shapefile
gbif_clipped_path = os.path.join(gbif_clipped_dir, "gbif_grasses_clipped.shp")
gbif_grasses_clipped.to_file(gbif_clipped_path)

print(f"Saved clipped GBIF shapefile to: {gbif_clipped_path}")

# Plot the clipped grasses and save the figure

In [ ]:
# plot clipped grass observations with the boundary
clipped_grass_points = gbif_grasses_clipped.hvplot(
    geo=True,
    tiles="EsriImagery",
    c="species",
    hover_cols=["species"],
    title="Clipped GBIF grass occurrences within Western Pawnee",
    frame_width=700,
    size=45
)

clipped_grass_boundary = master_bound_west_wgs84.hvplot(
    geo=True,
    color=None,
    line_color="yellow",
    line_width=2
)

clipped_grass_points * clipped_grass_boundary
